<img src = "https://miro.medium.com/max/748/1*wP8ubuQEIrtxtfd-DTOTig.jpeg" align = right width = 450>
<h1 align = left> Data 765: Python Fundamentals for Data Science</h1>
<h2 align = left> Lecture 11 | Pandas (II) </h2>
<h3 align = left> Yinxian Zhang | QC Sociology</h3>

# Table of Contents

<div class = "alert alert-info">

1. [Analyzing GSS with Pandas](#1)<br>
2. [Step 1: Read In and Inspect Data](#2)<br>
3. [Step 2: Getting Familiar with Your Data](#3)<br>
    3.1 [Basic descriptive statistics](#3.1)<br>
    3.2 [Missing values](#3.2)<br>
4. [Step 3: Data Transformations and Exporting](#4)<br>
    4.1 [Variable names and dtypes](#4.1)<br>
    4.2 [Recode variables](#4.2)<br>
    4.3 [Handle missing values](#4.3)<br>
    4.4 [Exporting data](#4.4)<br>

</div>
<hr>

We have learned about the data structures of `pandas` objects and the basic process of reading in and processing data. As we emphasized, the `pandas` library offers a full range of functionalities for data analysis, such that it is impossible to survey all methods and functions during this class. Instead, a better way to learn `pandas` (and virtually all new libraries) is to:

1. grasp the basic **structures of its objects**.
2. have a broad sense of its functionalities (what it can do).
3. understand the general **workflow** of working with this library. 
4. lastly, **learning by doing**. Seek help from Google or other online resources for specific functions/methods/tasks! 

We have talked about the first two points in the last session. 

In this session, I will walk you through **the trial-and-error, back-and-forth process** of dealing with real-life datasets, so that you can get a deepened understanding of the **data analytics workflow**. 

In [ ]:
# prepare packages 

import pandas as pd                
import numpy as np   

---

# Analyzing GSS with Pandas <a id=1></a>

The dataset we are going to use is one of the most classic datasets for sociology/social science -- [the General Social Survey (GSS)](http://gss.norc.org/About-The-GSS). 

> The (GSS) is a **nationally representative survey of adults in the United States** conducted **since 1972**. The GSS collects data on contemporary American society in order to monitor and explain trends in opinions, attitudes and behaviors. The GSS has adopted questions from earlier surveys which allows researchers to conduct comparisons for **up to 80 years.** <br>
<br>
The GSS contains a standard core of **demographic, behavioral, and attitudinal questions**, plus **topics of special interest**. Among the topics covered are civil liberties, crime and violence, intergroup tolerance, morality, national spending priorities, psychological well-being, social mobility, and stress and traumatic events. <br>
<br>
Altogether the GSS is the single best source for sociological and attitudinal trend data covering the United States. It allows researchers to examine the structure and functioning of society in general as well as the role played by relevant subgroups and to compare the United States to other nations. 

Again, this course assumes that you are familiar with survey data and basic statistical analysis (e.g., course content from Data  205 or Data 710/712). **Our primary goal is to put together all the Python tools we have covered and learn about the data analytic workflow in Python**. We would not go deep into any survey or statistical methods. For instance, We will not use *survey weights* in the exploratory analysis of the GSS, nor will we build *regression models* to analze the data. 

Once you get familiar with the workflow, you will be able to plug in what you've learned from other statistical courses and make full-fledge analyses of real-life datasets. 

# Step 1: Read in and Inspect Data   <a id=2></a>

Last time, we used `pandas` to read in a local Excel spreadsheet. Again, `pandas` supports a wide array of file formats, including Excel spreadsheets, CSV files, txt files, SQL, HTML, JSON, and even Stata and SPSS. Please [refer to this page](https://pandas.pydata.org/pandas-docs/stable/reference/io.html) for a full list.

Today, we will use the `read_csv` method to **read in the GSS data from online**, directly. Note that, being very flexible and powerful, `read_csv` features nearly 50 kwargs. You need to check out its full syntax and examples [from the documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).  

In [ ]:
path = 'https://raw.githubusercontent.com/UC-MACSS/persp-analysis/master/assignments/exploratory-data-analysis/data/gss2012.csv'

df = pd.read_csv(path, header=0)            # read data from online!

### Initial inspection

No matter what dataset(s) you are analyzing, the first couple things you need to do are almost the same -- to get a sense of the data.

In [ ]:
df.head()                             # returns the first 5 rows by default

There are A LOT of columns/variables -- 221. We can also see missing values `NaN`. To confirm the shape of the data:

In [ ]:
df.shape                                # returns (# of rows/obs, # of columns/variables)

Can we take a look at all of the 221 variables? 

In [ ]:
df.columns.values

There are a lot of variables, and most of the names are not intuitive -- however, this is **very common** for real-life datasets. You need to **get used to reading the codebook or documentation of the dataset, and find the variables that you need.** 

For the GSS dataset, you must have read the codebook available in BrightSpace.

Lastly, of course we'd like to use **the all-in-one command for dataset inspection:** 

In [ ]:
df.info()                                    # not showing the summary? Check docstring!

In [ ]:
df.info(verbose=True, null_counts=True)     
# returns information of each column, including # of non-missing values and dtypes.

### Limiting data to variables of interest (for large datasets)

My research interest is political sociology, and the main variables I would like to explore are people's political views (**this is the explanatory variable or independent variable**, i.e., `x`) and their voting decisions (**this is the outcome variable or dependent variable**, i.e. `y`). Except for the main variables, a typical sociological analysis also needs to include basic demographical information of the respondents as **control variables** (i.e. `z`s).

<p style='color:red'> Tips: save your cropped dataframe to another variable. **Do Not Overwrite** the orginal dataframe in case you need to use other variables later! </p>

In [ ]:
data = df[['pres08',                                     # dv
           'polviews',                                   # iv
           'id', 'age', 'race', 'educ'                   # controls           
        ]]

In [ ]:
# always remember to take a look at the dataframe itself

data.head()   

In [ ]:
# then obtain a summary using the all-in-one

data.info()              # note the many missing values in `pres08`

Now the dataset looks more managable. It's time for a more in-depth inspection. 

# Step 2: Getting Familiar with Your Data <a id=3></a>

After reading in data, you may want to ask questions like:

- are there any **missing values**? Would the missing values influence my analysis?
- are there any **illegal inputs** / special values that may affect my analysis? 
- are the variables in the desired **`dtype`s and formats**? 
- for **numerical variables**: what are the **measuring units**? are they discrete or continuous? 
- for **categorial variables**: how are they **coded**? are they **nominal or ordinal**?
- do I need to **recode** some of the variables?
- what are the **distributions** of the variables? Any **outliers or pecularities** that warrant further investigation?
- ...

Guided by these questions, you should investigate and, if needed, transform your data, preparing it for further analysis.

## Basic descriptive statistics    <a id=3.1></a>

<div class = "alert alert-info"> 
    A quick note: we used to use <code>df['col_name']</code> to index and slice columns. We can also use column names as attributes of the dataframe and call them by <code>df.col_name</code> (only applicable to column names without whitespaces or periods).  
</div>

### Numerical variables

In [1]:
data.age.describe()                          # negative values? over 200 yrs old? weird distributions?

NameError: name 'data' is not defined

### Categorical variables

#### First, find out what the categories are. 

In [ ]:
data.race.unique()                 

#### The `groupby` method
For categorical variables, knowing the specific "types" or "categories" of each variable is not enough. We also want to know more about the distribution.  An important function you will find useful is `groupby()`.

This is a powerful and flexible function that needs to be used with other methods or functions. Please [read more about `groupby()` here.](https://pandas.pydata.org/docs/user_guide/groupby.html)

In [ ]:
data.groupby('race')                   # returns an intermediate "groupby" object that is not "viewable"

# you can treat it as a sorted/grouped dataframe on the back-end of Python

In [ ]:
data.groupby('race').size()           # returns observation grouped by group/category

In [ ]:
data.groupby('polviews').size()       # not in order. warrants further transformation.

#### statistics of numerical variables grouped by categorical variables
After we inspect each individual variables, sometimes we also want to take a look at the simple bivariate or multivariate distributions of certain numerical variables over different categories (based on categorical variables). 

Since we only have one numerical `age` for this demo, let's do a "toy investigation" of age distribution over certain categories.   

In [ ]:
# recall that groupby object can be treated as a dataframe

data.groupby('race')['age'].mean()                      # avg age grouped by race        

In [ ]:
data.groupby(['pres08', 'race']).size()                 # distribution of the GSS sample over votes and race

 #### `groupby` with multiple aggregation statistics (`agg()`)

In [ ]:
data.groupby('polviews')['age'].agg([np.mean, np.median, np.max])    

This table may be hard to read, especially because one of the categorical variables have too many categories. Next week, we are going to learn about `pivot_table`, a pandas method that can give us a clearer, better organized view of the data. 

<div class = "alert alert-success">
<b> Class Quiz </b>
    
Let's start over! Please make another dataframe that only contains three variables from the <code>df</code> we read in. The three variables are <code>childs</code> (# of children), <code>marital</code> (marital status), and <code>degree</code> (highest degree obtained). <br>
    <br>
First, please describe the <code>childs</code> variable (giving me the descriptive statistics such as count, mean, std, min etc etc.) <br>
    <br>
Second, please obtain the average number of children of respondents with different degrees. I.e., the average number of children grouped by degree. 
    <br>
    
</div>

In [ ]:
## your code below




In [ ]:
import name
name.lucky(1, ignore=[])

---
Let's come back to our exploration of the `data` dataframe.

## Missing values    <a id=3.2></a>

Recall that when we used `df.info()` to inspect the dataframe, there are a lot of missing values in many variables. Let's take a closer look:

In [ ]:
data.isna().sum()                            # returns the number of missing values in the df

In [ ]:
data.pres08.isna().sum()                     # return the N of missing values for one variable only

In [ ]:
data[data.pres08.notna()]                    # limits to the subset *without* missing "pres08"

For certain analysis (such as correlations or regressions), `pandas` automatically removes missing values from relevant variables during the computation. Or, we can simply limit the dataset to a subset without missing values. 

<p style='color:red'>However, you need to be aware of the missing values in your dataset, and judge whether they will cause any problems for future analysis (for instance: are they random missing values, or does the missingness introduce any bias into the data?). Refer to research methods / survey design courses for details. </p>

# Step 3: Data Transformations and Exporting <a id=4></a>

Now that we've gotten a sense of the data and noted a few issues that need to be addressed, we should proceed to process and transform data. Let's start with the easiest thing first:

## Variables names and `dtype`s <a id=4.1></a>
First, you probably want to change variable names to their simpler forms:

In [ ]:
data.info()

In [ ]:
data = data.rename(columns={'pres08':'pres',
                            'polviews':'pol'})    # not really necessary in our case, for demo only
data.columns

We have a bunch of categorical variables and only one continuous variable `age`. However, `id` is recorded as `float`, which is problematic. Let's change it to `object`.

In [ ]:
data['id'] = data.id.astype('object')                       

In [ ]:
# double check if the transformation is sucessful:

data.info()

## Recode variables <a id=4.2></a>
### Ordered categorical variables

In [ ]:
data.pol.unique()

`pol`, our main variable of interest, is a categorical variable that features 7 categories ranging from "extremely liberal" to "extremely conservative". But the variable itself is not explicitly marked as an "ordinal variable", nor is it presenting the natural order of the variable. 

`pandas` allows you to specify the type (nominal v.s. ordinal) and order of a categorical variable for more fine-grain analysis: `pd.Categorical()` is the function that deals with categorical variables in pandas. You may want to check out the full syntax. 

In [ ]:
data.pol = pd.Categorical(data.pol, 
            # manually put in the right order
            categories=['ExtrmLib', 'Liberal', 'SlghtLib', 'Moderate', 'SlghtCons', 'Conserv', 'ExtrmCons'],   # ordered!
            # specify that it is an ordered (ordinal) variable
            ordered=True)                  

In [ ]:
data.pol.unique()

Behind the scene, **`pandas` assigns a numeric code to each category for ordering purposes.** So `ExtrmLib` may be 0, the "lowest" category, and `ExtrmCons` may be 6, the "highest" category. 

One thing to note: missing values `NaN` will be coded as `-1` in an ordinal variable!!

In [ ]:
data.pol                  # descriptive labels (what you see)

In [ ]:
data.pol.cat.codes       # numeric codes (what pandas uses)

In [ ]:
data.pol.cat.codes.unique()    # missing is coded as '-1' !

Other variables, such as `income` and `educ`, are also categorical ordinal variables. But since there are too many categories/levels (over 20), it makes sense to collapse them to fewer categories.

However, recoding these variables is more complicated than just specifying the variable type or its order. So let's save it for next week.  

## Handling missing/special values <a id=4.3></a>

One last thing for data preprocessing -- deal with missing or special values.

Sometimes, missing values or certain special values need to be handled properly, otherwise they may affect your data analysis. We usually have three different approaches to deal with missing/special values:

- Drop: Completely delete entire rows or columns with missing values or special values (nuclear solution, use with cautions);
- Selective Drop: Selectively remove missing values from certain rows or columns;
- Impute: Replace missing values (or a certain special values) with a value of our choice; 
- Replace special values: for instance, some survey data uses `999` to denote missing, so users need to replace `999` with `NaN`s. (This is not a problem for our dataset.)

#### 1. Drop:  remove entire rows or columns with any or all missing values (less common, use with cautions). 

In [ ]:
## completely remove the rows or columns with any or all missing values

df_copy = pd.read_csv(path, header=0)       # make a copy of original data
print(df_copy.shape)                        # check # of rows and colomns


df_drop_all = df_copy.dropna()              # by default, remove *any* rows with at least 1 missing values
print(df_drop_all.shape)                    # now any rows with missing values are removed.

#### 2. Selective Removal: remove only missing values in selected columns or rows (more common).

In [ ]:
# drop missing values in selected columns only

df_drop_select = df_copy.dropna(subset=['income06', 'educ'])

df_drop_select.shape

In [ ]:
# drop missing values only when the entire column or row is empty

df_drop_emptyrow = df_copy.dropna(how='all')
df_drop_emptycol = df_copy.dropna(axis=1, how='all')

print(df_drop_emptyrow.shape)
print(df_drop_emptycol.shape)

In [ ]:
# get full syntax and examples of `dropna()`
df_copy.dropna?

#### 3. Imputation: replace missing values with a value of our choice.

Caution: this move needs to be backed up by theories or solid justifications. The example below is just a toy demo without justifications.

For numerical variables, the most simple way to fill missing values is backward/forward filling (common for time-series data), or filling with the mean value. Statisticians have researched more complex strategies, but we won't go deep into this.

In [ ]:
data.age.isna().sum()

In [ ]:
## backward filling

data['age_imp1'] = data.age.fillna(method='backfill')  # methods for filling holes in the column 

# pad / ffill: propagate previous data forward to fill current missing value.
# backfill / bfill: use next data point to fill current missing value.

In [ ]:
data[['age', 'age_imp1']][data.age.isna()]

In [ ]:
## filling missing with the avg value

data.age.mean()

In [ ]:
data['age_imp2'] = data.age.fillna(48.193499238191976)

In [ ]:
data[['age', 'age_imp2']][data.age.isna()]

In [ ]:
data.age.fillna?

Sometimes imputation are more complicated than just backward/forward filling, or filling with a certain constant value. This is especially true when the variable is a categorical variable with many missing values. To address the issue of missingness, we may make theory-backed assumptions and **impute different values** depending on a set of conditions. We will talk more about this next week. 

## Exporting data <a id=4.4></a>

Finally, let's save the processed dataframe to a file!

In [ ]:
data.to_csv('processed_gss_data.csv', index=False)         # how to save it to an excel spreadsheet instead?

---
There may be a lot for you to process and digest. Again, the important thing is to grasp the data structures (`pd.Series` and `pd.DataFrame`)and the basic workflow of data analysis with `pandas`. Then for specific methods, you can always seek help from Google or other online resources. 

Regardless, please put in enough time and effort to practice `pandas`. The best way to learn it is learning by doing.

---
Copyright &copy; 2024 Yinxian Zhang | Department of Sociology | City University of New York, Queens College